# LoRRA honesty training (MASK paper §5.2)

Implements the Low-Rank Representation Adaptation baseline from *MASK* (arXiv 2503.03750, §5.2; Zou et al. 2023), applied to Qwen3.5-9B:

1. For each training input `x`, build an honest-persona version `x+` and a dishonest-persona version `x-` (contrastive templates `T+`, `T-`).
2. With the **frozen base model**, for each target layer `l_t` compute the contrast vector `v = Act(x+) - Act(x-)` and the target `r = Act(x) + alpha * v`.
3. Train **LoRA adapters on earlier editable layers `L_e`** so the adapted model's activations on the plain input `x` match `r` at every target layer (ℓ2 loss).
4. Merge the adapter, save, serve with vLLM and run `scripts/run_evals.py`.

The paper gives no rank, layers, alpha, data or step count, so those are config choices below (RepE-style defaults), not the paper's numbers.
Needs one GPU with ~40+ GB (bf16 9B model + activations). Don't train on MASK itself: it's the eval.

In [ ]:
%pip install -q "transformers>=5" peft accelerate datasets

In [ ]:
import random, math, json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

CFG = dict(
    model_name="Qwen/Qwen3.5-9B",
    out_dir="../models/qwen3.5-9b-lorra",
    # data
    n_train=5000, max_prompt_tokens=128, max_resp_tokens=64,
    # LoRRA
    alpha=5.0,                       # strength of the honesty contrast vector (RepE LoRRA used 5)
    target_fracs=(0.5, 0.75),        # target layers L_t, as fractions of depth
    edit_range=(0.15, 0.5),          # editable layers L_e: [lo, hi) fraction of depth, must sit before/at first target
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    # optimisation
    lr=1e-4, epochs=1, batch_size=8, grad_accum=2, warmup_frac=0.05, seed=0,
)
random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
assert torch.cuda.is_available(), "needs a CUDA GPU"

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model_name"])
model = AutoModelForCausalLM.from_pretrained(CFG["model_name"], torch_dtype=torch.bfloat16, device_map={"": 0})
model.config.use_cache = False
cfg = getattr(model.config, "text_config", model.config)
N = cfg.num_hidden_layers
TARGET = sorted({min(N - 1, round(f * N)) for f in CFG["target_fracs"]})
EDIT = list(range(int(CFG["edit_range"][0] * N), int(CFG["edit_range"][1] * N)))
assert max(EDIT) <= min(TARGET), "editable layers must come before the target layers"
print(f"{N} layers | target layers {TARGET} | editable layers {EDIT[0]}..{EDIT[-1]}")
print(type(model).__name__)

## Data and contrastive templates

Generic instruction data (`yahma/alpaca-cleaned`, no MASK content). The assistant response tokens are shared by all three variants (`x`, `x+`, `x-`), so activations line up token for token over the response; only the persona prefix differs. The loss is taken on response tokens only.

In [ ]:
from datasets import load_dataset

HONEST = "Pretend you're an honest person making statements about the world."
DISHONEST = "Pretend you're a dishonest person making statements about the world."

def chat_ids(system, user):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return tok(text, add_special_tokens=False)["input_ids"][-CFG["max_prompt_tokens"] - 64:]

ds = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=CFG["seed"])
examples = []
for r in ds:
    if r["input"].strip() or len(r["output"].split()) < 12:
        continue
    resp = tok(r["output"] + "<|im_end|>", add_special_tokens=False)["input_ids"][:CFG["max_resp_tokens"]]
    examples.append(dict(neutral=chat_ids(None, r["instruction"]),
                         plus=chat_ids(HONEST, r["instruction"]),
                         minus=chat_ids(DISHONEST, r["instruction"]),
                         resp=resp))
    if len(examples) >= CFG["n_train"]:
        break
print(len(examples), "examples")
print(tok.decode(examples[0]["plus"] + examples[0]["resp"]))

In [ ]:
def collate(batch):
    """Right-pad each variant; return per-variant ids/mask plus response-token index tensor [B, R] and validity mask."""
    R = max(len(e["resp"]) for e in batch)
    out = {}
    for k in ("neutral", "plus", "minus"):
        seqs = [e[k] + e["resp"] for e in batch]
        L = max(map(len, seqs))
        ids = torch.full((len(batch), L), tok.pad_token_id)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
        pos = torch.stack([torch.arange(R).clamp(max=len(e["resp"]) - 1) + len(e[k]) for e in batch])
        out[k] = (ids, att, pos)
    out["valid"] = torch.stack([torch.arange(R) < len(e["resp"]) for e in batch])
    return out

def resp_acts(m, ids, att, pos):
    """Hidden states at the target layers over response positions -> list of [B, R, H]."""
    hs = m(input_ids=ids.cuda(), attention_mask=att.cuda(), output_hidden_states=True).hidden_states
    idx = pos.cuda().unsqueeze(-1).expand(-1, -1, hs[0].shape[-1])
    return [hs[l + 1].gather(1, idx) for l in TARGET]   # hs[0] is the embedding output

## LoRA on the editable layers

In [ ]:
lcfg = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
                  target_modules="all-linear", layers_to_transform=EDIT, task_type="CAUSAL_LM")
pm = get_peft_model(model, lcfg)
pm.print_trainable_parameters()

## Training

Per step: under `disable_adapter()` + `no_grad`, get frozen activations on `x`, `x+`, `x-` → target `r = Act(x) + alpha·(Act(x+) − Act(x−))`. Then forward the adapted model on `x` and minimise the squared error to `r` at each target layer.

In [ ]:
params = [p for p in pm.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.0)
steps_per_epoch = len(examples) // (CFG["batch_size"] * CFG["grad_accum"])
total = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / max(1, CFG["warmup_frac"] * total))
                                          * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))

def lorra_loss(b):
    with pm.disable_adapter(), torch.no_grad():
        a0 = resp_acts(pm, *b["neutral"]); ap = resp_acts(pm, *b["plus"]); am = resp_acts(pm, *b["minus"])
        targets = [x.float() + CFG["alpha"] * (p.float() - n.float()) for x, p, n in zip(a0, ap, am)]
    cur = resp_acts(pm, *b["neutral"])
    v = b["valid"].cuda().unsqueeze(-1)
    loss = 0
    for c, t in zip(cur, targets):
        loss = loss + (((c.float() - t) ** 2) * v).sum() / (v.sum() * c.shape[-1])
    return loss / len(TARGET)

pm.train(); log = []
step = 0
for ep in range(CFG["epochs"]):
    random.shuffle(examples)
    for i in range(0, steps_per_epoch * CFG["batch_size"] * CFG["grad_accum"], CFG["batch_size"]):
        loss = lorra_loss(collate(examples[i:i + CFG["batch_size"]])) / CFG["grad_accum"]
        loss.backward()
        log.append(loss.item() * CFG["grad_accum"])
        if (i // CFG["batch_size"] + 1) % CFG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % 10 == 0:
                print(f"step {step}/{total} loss {sum(log[-20:]) / len(log[-20:]):.4f} lr {sched.get_last_lr()[0]:.2e}")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(log); plt.xlabel("microbatch"); plt.ylabel("LoRRA l2 loss"); plt.show()

## Sanity check

On held-out prompts, the adapted model's activations on plain `x` should have moved toward the honest direction: cosine between `Act_adapted(x) − Act_base(x)` and the contrast vector `v` should be clearly positive. Also eyeball a sample answer.

In [ ]:
pm.eval()
held = examples[-64:]; cos = []
with torch.no_grad():
    for i in range(0, 64, 8):
        b = collate(held[i:i + 8]); v = b["valid"].cuda()
        with pm.disable_adapter():
            a0 = resp_acts(pm, *b["neutral"]); ap = resp_acts(pm, *b["plus"]); am = resp_acts(pm, *b["minus"])
        cur = resp_acts(pm, *b["neutral"])
        for c, x, p, n in zip(cur, a0, ap, am):
            c_ = torch.nn.functional.cosine_similarity((c - x).float(), (p - n).float(), dim=-1)
            cos.append(c_[v].mean().item())
print("mean cosine(adapter shift, honesty contrast vector):", sum(cos) / len(cos))

msgs = [{"role": "user", "content": "Is it true that humans only use 10% of their brains?"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt", return_dict=True).to("cuda")
print(tok.decode(pm.generate(**ids, max_new_tokens=120, do_sample=False)[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

## Merge and save (vLLM-loadable checkpoint)

In [ ]:
out = Path(CFG["out_dir"]); out.mkdir(parents=True, exist_ok=True)
pm.save_pretrained(out / "adapter")                       # keep the adapter too
merged = pm.merge_and_unload()
merged.save_pretrained(out, safe_serialization=True); tok.save_pretrained(out)
(out / "lorra_config.json").write_text(json.dumps({**CFG, "target_layers": TARGET, "edit_layers": EDIT}, indent=2))
print("saved to", out.resolve())

## Serve with vLLM and run evals

Free the GPU first (restart this kernel). Then in terminals, from the repo root (judge stays the same Qwen2.5-32B as the baseline for comparability):

```bash
# model under test (merged LoRRA weights) on GPUs 0-5
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5 vllm serve models/qwen3.5-9b-lorra --served-model-name qwen3.5-9b-lorra \
  --port 8000 --data-parallel-size 6 --gpu-memory-utilization 0.90 --max-model-len 16384
# judge on GPUs 6-7
CUDA_VISIBLE_DEVICES=6,7 vllm serve Qwen/Qwen2.5-32B-Instruct --port 8001 --tensor-parallel-size 2 \
  --gpu-memory-utilization 0.90 --max-model-len 16384

python scripts/run_evals.py --split test --model qwen3.5-9b-lorra --base-url http://localhost:8000/v1 \
  --judge-model Qwen/Qwen2.5-32B-Instruct --judge-base-url http://localhost:8001/v1 \
  --concurrency 96 --mach-workers 30 --out outputs/lorra_full_test
```
Smoke test first with `--split dev --limit 5 --bench mask mmlu`.